# NSL-KDD Dataset Exploratory Data Analysis

This notebook conducts a detailed Exploratory Data Analysis (EDA) on the NSL-KDD intrusion detection dataset. We analyze dataset structures, feature relationships, class distributions, and identify implications for subsequent feature selection operations.

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.nsl_kdd import NSLKDDLoader

# Load configuration
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

nsl_train_path = config["data"]["nsl_kdd_train"]
print(f"Target train path configured: {nsl_train_path}")

In [ ]:
# Load data using NSLKDDLoader
loader = NSLKDDLoader()
train_df = loader.load(nsl_train_path)
print(f"Loaded DataFrame shape: {train_df.shape}")

In [ ]:
# Dataset shape and class distribution
print(f"Dataset Rows: {train_df.shape[0]}, Columns: {train_df.shape[1]}")

# Get mapped classes
X, y = loader.preprocess(train_df)
class_counts = pd.Series(y).value_counts().sort_index()
class_names = loader.classes

plt.figure(figsize=(8, 5))
sns.barplot(x=[class_names[i] for i in class_counts.index], y=class_counts.values, palette="viridis")
plt.title("NSL-KDD 5-Class Label Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--")
plt.savefig("figures/class_distribution.png")
plt.show()

In [ ]:
# Feature correlation heatmap on continuous features
num_features = train_df.select_dtypes(include=[np.number])
# Limit to first 15 columns for visual clarity
corr_matrix = num_features.iloc[:, :15].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix for Selected Continuous Features")
plt.savefig("figures/feature_correlation.png")
plt.show()

In [ ]:
# Attack category distribution pie chart
plt.figure(figsize=(8, 8))
plt.pie(class_counts.values, labels=[class_names[i] for i in class_counts.index], autopct="%1.1f%%", startangle=140, colors=sns.color_palette("pastel"))
plt.title("Attack Class Proportions")
plt.savefig("figures/attack_pie_chart.png")
plt.show()

In [ ]:
# Feature statistics table
summary_stats = train_df.describe().T
print("Feature Summary Statistics (First 10 Features):")
print(summary_stats.head(10))

In [ ]:
# Missing value analysis
missing_values = train_df.isnull().sum()
print(f"Total Missing Values: {missing_values.sum()}")

### Key Observations and Implications

1. **Class Balance**: The normal class and DoS attacks represent the vast majority of records, whereas U2R and R2L classes are extremely rare. This imbalance requires stratified splitting during validation.
2. **Feature Redundancy**: The correlation analysis shows high collinearity between several traffic metrics (e.g., serror_rate and srv_serror_rate), highlighting the clear necessity of BWOA feature selection to reduce redundancy.
3. **Zero Variance**: Certain columns (like num_outbound_cmds) have zero variance and should be trimmed to prevent numeric issues.